# SOFR, FX & Cross-Currency Curves · Episode 3 — The SOFR curve

Companion notebook for the video. We bootstrap a SOFR discount curve from overnight index swaps and three-month SOFR futures, check it by hand, and look at what it says about future rates. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Data.** All quotes in `quotes_sofr_illustrative.csv` are illustrative, made up for teaching: shaped like the market in late September 2026, but not market data. Educational material only, not investment advice.

1. Setup · 2. Conventions · 3. SOFR futures · 4. Quotes · 5. Bootstrap · 6. Hand checks · 7. Reprice · 8. A five-year swap · 9. The curve · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_sofr_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_sofr_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_sofr_illustrative.csv').write_text("""instrument,label,value,unit
ois,1W,3.88,pct
ois,1M,3.87,pct
ois,2M,3.86,pct
ois,3M,3.85,pct
future,SR3 Dec 2026,96.20,price
future,SR3 Mar 2027,96.30,price
future,SR3 Jun 2027,96.39,price
future,SR3 Sep 2027,96.44,price
ois,2Y,3.66,pct
ois,3Y,3.63,pct
ois,5Y,3.63,pct
ois,7Y,3.70,pct
ois,10Y,3.81,pct
ois,15Y,3.94,pct
ois,20Y,3.98,pct
ois,30Y,3.96,pct
""")
print('wrote quotes_sofr_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-25"
quotes_file = "quotes_sofr_illustrative.csv"
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
iso = lambda d: d.ISO()
out = {"meta": {"series": "SOFR, FX & Cross-Currency Curves", "episode": 3, "valuation_date": valuation_date,
                "quantlib_version": ql.__version__, "quotes_label": "illustrative (made up for teaching)",
                "generated_at": dt.datetime.now().isoformat(timespec="seconds")}}
print("valuation date", today, "| QuantLib", ql.__version__)

The shared SOFR curve code (also used by later episodes):

In [ ]:
# SOFR curve used from Series 2 Episode 3 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today (the evaluation date) from the setup cell.
#   SOFR OIS: CFTC Swaps Made Available to Trade chart (USD SOFR OIS): spot T+2, annual fixed and floating
#     legs, ACT/360, payment lag 2 days, New York business days; SOFR fixes on the government securities calendar.
#   3M SOFR futures: CME Submission 18-069: SOFR compounded over the IMM-to-IMM Reference Quarter,
#     price = 100 - rate; named by the month the Reference Quarter starts. Convexity set to zero (a simplification).
USD_CAL = ql.UnitedStates(ql.UnitedStates.Settlement)  # stand-in for New York business days
USD_DC = ql.Actual360()
MONTHS = {m: i for i, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

def sofr_ois_helper(tenor, quote):
    return ql.OISRateHelper(2, ql.Period(tenor), quote, ql.Sofr(), paymentLag=2,
                            paymentFrequency=ql.Annual, paymentCalendar=USD_CAL,
                            convention=ql.ModifiedFollowing, endOfMonth=True)

def build_sofr_curve(quotes):
    """quotes: rows (instrument, label, value); OIS values in %, futures as prices. Returns curve, quotes, helpers."""
    q, helpers = {}, {}
    for r in quotes.itertuples():
        if r.instrument == "future":  # label like "SR3 Dec 2026"
            q[r.label] = ql.SimpleQuote(r.value)
            _, mon, year = r.label.split()
            helpers[r.label] = ql.SofrFutureRateHelper(ql.QuoteHandle(q[r.label]), MONTHS[mon], int(year),
                                                       ql.Quarterly)
        else:
            q[r.label] = ql.SimpleQuote(r.value / 100)
            helpers[r.label] = sofr_ois_helper(r.label, ql.QuoteHandle(q[r.label]))
    curve = ql.PiecewiseLogLinearDiscount(today, list(helpers.values()), USD_DC)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now: the helpers only see the curve once it has been built
    return curve, q, helpers

## 2. Conventions

| | USD SOFR OIS | AUD AONIA OIS (Series 1) |
|---|---|---|
| Start | two business days after the trade (T+2) | next business day (T+1) |
| Day count | ACT/360, both legs | ACT/365 Fixed, both legs |
| Payments | annual, both legs | at maturity up to 12 months; then annual after a front stub |
| Payment lag | two business days | two business days |
| Fixing calendar | US government securities | Sydney |

Sources: CFTC *Swaps Made Available to Trade* (USD SOFR OIS); AFMA *Interest Rate Derivative Conventions* (June 2025) §3.7, §3.17, §5.2.

In [ ]:
spot = USD_CAL.advance(today, 2, ql.Days)
out["conventions"] = {
    "spot": iso(spot),
    "rows": [
        {"item": "Start", "usd": "T+2", "aud": "T+1"},
        {"item": "Day count", "usd": "ACT/360", "aud": "ACT/365F"},
        {"item": "Payments", "usd": "Annual", "aud": "Annual, front stub"},
        {"item": "Payment lag", "usd": "2 days", "aud": "2 days"},
        {"item": "Fixing calendar", "usd": "US govt securities", "aud": "Sydney"},
    ],
}
pd.DataFrame(out["conventions"]["rows"])

## 3. SOFR futures

From CME's listing filing for Three-Month SOFR Futures (CME Submission 18-069 to the CFTC, 2018):

* **Reference Quarter**: from (and including) the third Wednesday of the contract month to (but not including) the third Wednesday of the delivery month, three months later. The contract is named by the month its Reference Quarter starts.
* **Final settlement**: $R = \left[\prod_i \left(1 + \frac{d_i}{360}\,\frac{r_i}{100}\right) - 1\right] \times \frac{360}{D} \times 100$, SOFR compounded over the Reference Quarter; price $= 100 - R$.
* **Size**: each basis point of $R$ is worth USD 25 per contract.

Futures settle every day, which makes them differ slightly from swaps (a *convexity* difference). Here we set that difference to zero: a simplification.

In [ ]:
quotes = pd.read_csv(quotes_file)
futs = quotes[quotes.instrument == "future"].copy()
fut_rows = []
for r in futs.itertuples():
    _, mon, year = r.label.split()
    start = ql.IMM.nextDate(ql.Date(1, MONTHS[mon], int(year)))
    end = ql.IMM.nextDate(start + ql.Period(2, ql.Months))
    fut_rows.append({"label": r.label, "price": r.value, "rate_pct": 100 - r.value,
                     "start": iso(start), "end": iso(end), "days": end - start})
out["futures"] = {"rows": fut_rows, "bp_value_usd": 25}
pd.DataFrame(fut_rows)

## 4. Quotes (illustrative)

In [ ]:
def show(r):
    return f"{r.value:.2f}" if r.instrument == "future" else f"{r.value:.2f}%"
quotes["quote"] = [show(r) for r in quotes.itertuples()]
quotes["kind"] = quotes.instrument.map({"ois": "SOFR OIS", "future": "3M SOFR future"})
short = quotes[~quotes.label.str.endswith("Y")].reset_index(drop=True)   # 1W-3M swaps and the futures
long_ = quotes[quotes.label.str.endswith("Y")].reset_index(drop=True)    # 2Y-30Y swaps
wide = [{"a": a.label, "a_kind": a.kind, "a_quote": a.quote, "b": b.label, "b_quote": b.quote}
        for a, b in zip(short.itertuples(), long_.itertuples())]
out["quotes"] = {"rows": quotes[["label", "kind", "quote"]].to_dict("records"), "wide": wide,
                 "n": len(quotes), "n_ois": int((quotes.instrument == "ois").sum()),
                 "n_futures": int((quotes.instrument == "future").sum())}
quotes[["label", "kind", "quote"]]

## 5. Bootstrap

Each helper turns one quote into one unknown discount factor, shortest first. Log-linear interpolation on discount factors (piecewise-flat forwards).

In [ ]:
curve, q, helpers = build_sofr_curve(quotes)
dc365 = ql.Actual365Fixed()
pillars = pd.DataFrame([{
    "label": k, "pillar": iso(h.pillarDate()), "t_years": dc365.yearFraction(today, h.pillarDate()),
    "discount_factor": curve.discount(h.pillarDate()),
    "zero_pct": curve.zeroRate(h.pillarDate(), dc365, ql.Continuous).rate() * 100,
} for k, h in helpers.items()])
out["pillars"] = pillars.to_dict("records")
pillars

## 6. Hand checks

**A one-period OIS.** For a swap with a single period from $s$ to $e$, both legs pay on the same date, so the payment factor cancels and $P(e)/P(s) = 1/(1 + K\tau)$ with $\tau$ = days/360.

**A future.** Compounding the curve's daily forward SOFR over the Reference Quarter with CME's formula gives the futures rate; each daily factor is $P(d_i)/P(d_{i+1})$, so the product is just $P(\text{start})/P(\text{end})$.

In [ ]:
h1m = helpers["1M"]
s, e = h1m.swap().startDate(), h1m.swap().maturityDate()
K = q["1M"].value()
tau = (e - s) / 360
ois = {"start": iso(s), "end": iso(e), "days": e - s, "rate_pct": K * 100,
       "ratio_hand": 1 / (1 + K * tau), "ratio_curve": curve.discount(e) / curve.discount(s)}
ois["abs_diff"] = abs(ois["ratio_hand"] - ois["ratio_curve"])

f = fut_rows[0]
fs, fe = ql.DateParser.parseISO(f["start"]), ql.DateParser.parseISO(f["end"])
fix_cal = ql.Sofr().fixingCalendar()
prod, d = 1.0, fs
while d < fe:
    nxt = min(fix_cal.advance(d, 1, ql.Days), fe)
    r_i = (curve.discount(d) / curve.discount(nxt) - 1) * 360 / (nxt - d)   # forward SOFR for that day
    prod *= 1 + r_i * (nxt - d) / 360
    d = nxt
fut = {"label": f["label"], "price": f["price"], "rate_from_price_pct": f["rate_pct"],
       "rate_cme_formula_pct": (prod - 1) * 360 / (fe - fs) * 100,
       "rate_df_ratio_pct": (curve.discount(fs) / curve.discount(fe) - 1) * 360 / (fe - fs) * 100,
       "price_quantlib": helpers[f["label"]].impliedQuote()}
fut["abs_diff_bp"] = abs(fut["rate_cme_formula_pct"] - fut["rate_from_price_pct"]) * 100
out["hand"] = {"ois_1m": ois, "future": fut}
pd.DataFrame({"1M OIS": pd.Series(ois), "future": pd.Series(fut)})

## 7. Reprice every quote

In [ ]:
rep = pd.DataFrame([{"label": k, "quote": q[k].value(), "implied": h.impliedQuote(),
                     "error": h.impliedQuote() - q[k].value()} for k, h in helpers.items()])
rep["error_bp"] = [e * 1e4 if k in quotes[quotes.instrument == "ois"].label.values else e * 100
                   for k, e in zip(rep.label, rep.error)]
out["reprice"] = {"max_abs_error_bp": float(rep.error_bp.abs().max())}
print("max |error| (bp):", out["reprice"]["max_abs_error_bp"])
rep

## 8. A five-year SOFR swap, by hand

Annual periods, each paid two business days after it ends. With the payment lag the floating leg does not telescope, so value each period: forward SOFR compounded over the period, $P(s_i)/P(e_i) - 1$, paid at $p_i$. The par rate is

$$K = \frac{\sum_i \big(P(s_i)/P(e_i) - 1\big)\,P(p_i)}{\sum_i \tau_i\,P(p_i)}, \qquad \tau_i = \frac{\text{days}(s_i, e_i)}{360}$$

In [ ]:
h5 = helpers["5Y"]
sw = h5.swap()
float_pv, annuity, periods = 0.0, 0.0, []
for cf in sw.fixedLeg():
    c = ql.as_coupon(cf)
    si, ei, pi = c.accrualStartDate(), c.accrualEndDate(), c.date()
    fwd = curve.discount(si) / curve.discount(ei) - 1
    float_pv += fwd * curve.discount(pi)
    annuity += c.accrualPeriod() * curve.discount(pi)
    periods.append({"start": iso(si), "end": iso(ei), "pay": iso(pi), "tau": c.accrualPeriod(),
                    "fwd_rate_pct": fwd / c.accrualPeriod() * 100, "df_pay": curve.discount(pi)})
par5 = {"par_hand_pct": float_pv / annuity * 100, "par_quantlib_pct": h5.impliedQuote() * 100,
        "quote_pct": q["5Y"].value() * 100, "annuity": annuity, "periods": periods,
        "notional": 100_000_000, "dv01_per_100m": annuity * 1e8 * 1e-4}
par5["abs_diff_bp"] = abs(par5["par_hand_pct"] - par5["par_quantlib_pct"]) * 100
out["swap5y"] = par5
pd.DataFrame(periods)

## 9. The curve

Zero rates (continuous, ACT/365F, for comparison with Series 1) and 3-month forward SOFR rates (simple, ACT/360) every month for thirty years.

In [ ]:
grid = []
for m in range(0, 361, 1):
    d0 = USD_CAL.advance(spot, m, ql.Months)
    d1 = USD_CAL.advance(d0, 3, ql.Months)
    grid.append({"t": dc365.yearFraction(today, d0),
                 "zero_pct": curve.zeroRate(d0, dc365, ql.Continuous).rate() * 100 if m else None,
                 "fwd3m_pct": curve.forwardRate(d0, d1, USD_DC, ql.Simple).rate() * 100})
g = pd.DataFrame(grid)
g.loc[0, "zero_pct"] = g.zero_pct.iloc[1]
lo = g.loc[g.t <= 3, "fwd3m_pct"]
out["chart"] = {"t": g.t.round(4).tolist(), "zero_pct": g.zero_pct.round(4).tolist(),
                "fwd3m_pct": g.fwd3m_pct.round(4).tolist(),
                "pillar_t": pillars.t_years.round(4).tolist(), "pillar_zero": pillars.zero_pct.round(4).tolist(),
                "ticks": [{"x": float(y), "label": f"{y}Y"} for y in (0, 2, 5, 10, 15, 20, 30)]}
out["shape"] = {"fwd_now_pct": float(g.fwd3m_pct.iloc[0]), "fwd_low_pct": float(lo.min()),
                "fwd_low_t": float(g.t[lo.idxmin()]), "fwd_10y_pct": float(g.loc[(g.t - 10).abs().idxmin(), "fwd3m_pct"]),
                "zero_1y_pct": float(g.loc[(g.t - 1).abs().idxmin(), "zero_pct"]),
                "zero_30y_pct": float(g.zero_pct.iloc[-1])}
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(g.t, g.zero_pct, label="zero rate (cont., ACT/365F)")
ax.step(g.t, g.fwd3m_pct, where="post", label="3M forward SOFR (ACT/360)")
ax.scatter(pillars.t_years, pillars.zero_pct, s=14, color="k", zorder=3, label="pillars")
ax.set(xlabel="years", ylabel="%"); ax.grid(alpha=.3); ax.legend()
pd.Series(out["shape"])

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")